In [14]:
import boto3
import lazycogs
from odc.geo.geobox import GeoBox
from affine import Affine
import rustac
from obstore.store import S3Store

import os

os.environ["AWS_PROFILE"] = "cdse"

geobox = GeoBox(
    shape=(10, 10),
    affine=Affine(5000.0, 0.0, 690400.0, 0.0, -5000.0, 4660400.0),
    crs="EPSG:32614"
)

# 1. Fetch the frozen credentials directly from boto3
session = boto3.Session(profile_name="cdse")
creds = session.get_credentials().get_frozen_credentials()

# # 2. Build the configuration dictionary for obstore
# obstore_config = {
#     "aws_access_key_id": creds.access_key,
#     "aws_secret_access_key": creds.secret_key,
#     # "aws_endpoint_url": "https://eodata.dataspace.copernicus.eu",
#     "aws_virtual_hosted_style_request": "false",
#     "aws_region": session.region_name or "default",
# }
# # If your profile uses temporary STS credentials, pass the token too
# if creds.token:
#     obstore_config["aws_session_token"] = creds.token

obstore_config = {
    "access_key_id": creds.access_key,
    "secret_access_key": creds.secret_key,
    "endpoint": "https://eodata.dataspace.copernicus.eu",  # Required for CDSE
    "virtual_hosted_style_request": "false",
    "region": session.region_name or "default",
}

if creds.token:
    obstore_config["token"] = creds.token



PARQUET = "items.parquet"
items = await rustac.search(
        # PARQUET,
        href="https://stac.dataspace.copernicus.eu/v1/",
        collections=["clms_lst_global_3km_hourly_v3_cog"],
        datetime="2021-03-27",
        bbox=list(geobox.boundingbox.to_crs(4326)),
        limit=1,
    )

store = S3Store(
    bucket="eodata",
    config=obstore_config,
)

In [ ]:
from async_geotiff import GeoTIFF
from urllib.parse import urlparse

def strip_bucket(href: str) -> str:
    # href: https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/path/to/file.tif
    # store is rooted at the bucket, so the path is just path/to/file.tif
    return urlparse(href).path.lstrip("/").removeprefix("eodata/")



await lazycogs.open_cog(strip_bucket(items[0]["assets"]["lst_lst"]["href"]), store=store)

<xarray.DataArray (band: 1, y: 5601, x: 14400)> Size: 161MB
array([[[-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        ...,
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999]]],
      shape=(1, 5601, 14400), dtype=int16)
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 45kB 70.0 69.97 69.95 69.92 ... -69.95 -69.98 -70.0
  * x            (x) float64 115kB -180.0 -180.0 -179.9 ... 179.9 180.0 180.0
    spatial_ref  int64 8B 0
Indexes:
  ┌ x        RasterIndex (crs=EPSG:4326)
  └ y
Attributes:
    grid_mapping:  spatial_ref
    _FillValue:    -9999.0
    scale_factor:  0.01
    add_offset:    273.15

In [7]:
items[0]["assets"]

{'Product': {'href': 'https://download.dataspace.copernicus.eu/odata/v1/Products(db6d312c-12bf-49f8-9fec-424bf08a4ffb)/$value',
  'file:size': 25720143,
  'file:checksum': 'd50110dcddac5dc58d26bddaa50e251a6f5d1c',
  'file:local_path': 'c_gls_LST_202103272300_GLOBE_GEO_V3.0.1_cog.zip',
  'type': 'application/zip',
  'roles': ['data', 'metadata', 'archive'],
  'title': 'Zipped product',
  'auth:refs': ['oidc']},
 'lst_lst': {'href': 's3://eodata/CLMS/bio-geophysical/land_surface_temperature/lst_global_3km_hourly_v3/2021/03/27/c_gls_LST_202103272300_GLOBE_GEO_V3.0.1_cog/c_gls_LST-LST_202103272300_GLOBE_GEO_V3.0.1.tiff',
  'alternate': {'https': {'href': 'https://download.dataspace.copernicus.eu/odata/v1/Products(db6d312c-12bf-49f8-9fec-424bf08a4ffb)/Nodes(c_gls_LST_202103272300_GLOBE_GEO_V3.0.1_cog)/Nodes(c_gls_LST-LST_202103272300_GLOBE_GEO_V3.0.1.tiff)/$value',
    'auth:refs': ['oidc'],
    'storage:refs': [],
    'alternate:name': 'HTTPS'}},
  'file:size': 14369966,
  'proj:shape': [5

In [8]:
lazycogs.open_item(items[0], bands=["lst_lst", "lst_qflag"], store=store)

<xarray.DataArray (band: 2, y: 5601, x: 14400)> Size: 323MB
array([[[-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        ...,
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999]],

       [[    0,     0,     0, ...,     0,     0,     0],
        [    0,     0,     0, ...,     0,     0,     0],
        [    0,     0,     0, ...,     0,     0,     0],
        ...,
        [    0,     0,     0, ...,     0,     0,     0],
        [    0,     0,     0, ...,     0,     0,     0],
        [    0,     0,     0, ...,     0,     0,     0]]],
      shape=(2, 5601, 14400), dtype=int16)
Coordinates:
  * band         (band) <U9 72B 'lst_lst' 'lst_qflag'
  * y            (y) float64 45kB 70.0 69.97 69.95 69.92 ... -69.95 -69.98 -70.0
  * x            (x) float64 115kB -180.0 -180.0 -179.9 ... 179.9 180.0 180.0
    spatial_ref  int64 8B 0
Indexes:
  ┌ x        RasterIndex (crs=EPSG:4326)
  └ y
Attributes:
    grid_mapping:  spatial_ref
    _FillValue:    -9999.0
    scale_factor:  0.01
    add_offset:    273.15

In [10]:
items[0]

{'type': 'Feature',
 'stac_version': '1.0.0',
 'id': 'S2A_31UFT_20230624_0_L2A',
 'properties': {'created': '2023-06-24T21:50:58.073Z',
  'platform': 'sentinel-2a',
  'constellation': 'sentinel-2',
  'instruments': ['msi'],
  'eo:cloud_cover': 2.253168,
  'proj:epsg': 32631,
  'mgrs:utm_zone': 31,
  'mgrs:latitude_band': 'U',
  'mgrs:grid_square': 'FT',
  'grid:code': 'MGRS-31UFT',
  'view:sun_azimuth': 158.945977765612,
  'view:sun_elevation': 60.30393659593019,
  's2:degraded_msi_data_percentage': 0.0293,
  's2:nodata_pixel_percentage': 1e-05,
  's2:saturated_defective_pixel_percentage': 0,
  's2:dark_features_percentage': 0.000236,
  's2:cloud_shadow_percentage': 2.191164,
  's2:vegetation_percentage': 74.626517,
  's2:not_vegetated_percentage': 17.30607,
  's2:water_percentage': 3.063995,
  's2:unclassified_percentage': 0.558097,
  's2:medium_proba_clouds_percentage': 1.142817,
  's2:high_proba_clouds_percentage': 1.108019,
  's2:thin_cirrus_percentage': 0.002332,
  's2:snow_ice_pe

In [9]:
import rustac
import lazycogs

# 1. Find one low-cloud Sentinel-2 scene (public STAC API, no credentials).
items = await rustac.search(
    href="https://earth-search.aws.element84.com/v1",
    collections=["sentinel-2-l2a"],
    bbox=[4.8, 52.3, 5.0, 52.5],          # Amsterdam
    datetime="2023-06-01/2023-06-30",
    query={"eo:cloud_cover": {"lt": 10}},
    limit=1,
)

# 2. Stack the 10 m bands at native resolution → one (band, y, x) DataArray,
#    band labelled by asset key. The asset hrefs point at the public
#    sentinel-cogs bucket over https, so no store= is needed.
da = lazycogs.open_item(items[0], bands=["red", "green", "blue", "nir"])

print(da.dims, dict(da.sizes))     # ('band', 'y', 'x') {'band': 4, 'y': 10980, 'x': 10980}
print(da["band"].values)           # ['red' 'green' 'blue' 'nir']


AsyncTiffException: GenericError: Generic S3 error: Error performing PUT http://169.254.169.254/latest/api/token in 15.25687525s, after 10 retries, max_retries: 10, retry_timeout: 180s  - HTTP error: error sending request

Debug source:
Generic {
    store: "S3",
    source: RetryError(
        RetryErrorImpl {
            method: PUT,
            uri: Some(
                http://169.254.169.254/latest/api/token,
            ),
            retries: 10,
            max_retries: 10,
            elapsed: 15.25687525s,
            retry_timeout: 180s,
            inner: Http(
                HttpError {
                    kind: Connect,
                    source: reqwest::Error {
                        kind: Request,
                        source: hyper_util::client::legacy::Error(
                            Connect,
                            ConnectError(
                                "tcp connect error",
                                169.254.169.254:80,
                      